# Performance Metrics

This notebook computes the evaluation metrics of the OTSU-GWWHO framework. Segmentation quality is measured with the Jaccard Similarity Coefficient (JSC), Dice Similarity Coefficient (DSC) and Pixel Accuracy (PA). Classification is evaluated on the held-out test set with Precision (PPV), Sensitivity / Recall (SEN), Specificity (SPC), F1-score, Accuracy (ACC) and error rate, derived from the confusion matrix (one-vs-rest Tp, Tn, Fp, Fn). 95% confidence intervals are estimated with bootstrap resampling.

## Prerequisites

- Python 3.10+
- Required libraries: `numpy`, `opencv-python`, `pandas`, `scikit-learn`, `matplotlib`
- Test predictions from `VGG16_DCNN_classification.ipynb`
- Ground-truth masks for segmentation evaluation (`GT_MASK_DIR`)

## Workflow

1. Compute JSC, DSC and PA between the OTSU-GWWHO masks and the ground-truth masks
2. Build the confusion matrix from the test predictions
3. Compute PPV, SEN, SPC, F1, ACC and error
4. Estimate 95% confidence intervals with bootstrap resampling
5. Plot and save the confusion matrix and metric tables

In [ ]:
# ==========================================================
# IMPORTS & CONFIGURATION
# ==========================================================

import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

PRED_MASK_DIR = "./outputs/04_segmented/test/mask"
GT_MASK_DIR = "./ground_truth_masks"            # same file names / class sub-folders as PRED_MASK_DIR
PRED_CSV = "./outputs/07_results/test_predictions_ensemble.csv"
RESULTS_DIR = "./outputs/07_results"
CLASS_NAMES = ["MD", "MOD", "ND", "VMD"]
BOOTSTRAP_ITERS = 1000
RANDOM_SEED = 42

os.makedirs(RESULTS_DIR, exist_ok=True)

## Segmentation Metrics

JSC = |PS ∩ GS| / |PS ∪ GS|, DSC = 2|PS ∩ GS| / (|PS| + |GS|), PA = correctly classified pixels / total pixels.

In [ ]:
# ==========================================================
# SEGMENTATION METRICS (JSC, DSC, PA)
# ==========================================================

def jaccard(ps, gs):
    union = np.logical_or(ps, gs).sum()
    return np.logical_and(ps, gs).sum() / union if union else 1.0


def dice(ps, gs):
    total = ps.sum() + gs.sum()
    return 2 * np.logical_and(ps, gs).sum() / total if total else 1.0


def pixel_accuracy(ps, gs):
    return (ps == gs).mean()


seg_rows = []
for pred_path in sorted(glob.glob(os.path.join(PRED_MASK_DIR, "**", "*.png"), recursive=True)):
    gt_path = os.path.join(GT_MASK_DIR, os.path.relpath(pred_path, PRED_MASK_DIR))
    if not os.path.exists(gt_path):
        continue
    ps = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE) > 127
    gs = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE) > 127
    seg_rows.append({"image": pred_path, "JSC": jaccard(ps, gs),
                     "DSC": dice(ps, gs), "PA": pixel_accuracy(ps, gs)})

seg_df = pd.DataFrame(seg_rows)
if len(seg_df):
    seg_df.to_csv(os.path.join(RESULTS_DIR, "segmentation_metrics.csv"), index=False)
    print(seg_df[["JSC", "DSC", "PA"]].mean().round(4))
else:
    print("No ground-truth masks found in", GT_MASK_DIR)

## Classification Metrics

PPV = Tp/(Tp+Fp), SEN = Tp/(Tp+Fn), SPC = Tn/(Tn+Fp), F1 = 2Tp/(2Tp+Fp+Fn), ACC = (Tp+Tn)/(Tp+Tn+Fp+Fn). PPV, SEN, SPC and F1 are macro-averaged over the four classes, and ACC is the overall accuracy.

In [ ]:
# ==========================================================
# CLASSIFICATION METRICS
# ==========================================================

def metrics_from_cm(cm):
    cm = cm.astype(np.float64)
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    tn = cm.sum() - tp - fp - fn
    with np.errstate(divide="ignore", invalid="ignore"):
        ppv = np.nan_to_num(tp / (tp + fp))
        sen = np.nan_to_num(tp / (tp + fn))
        spc = np.nan_to_num(tn / (tn + fp))
        f1 = np.nan_to_num(2 * tp / (2 * tp + fp + fn))
    acc = tp.sum() / cm.sum()
    per_class = pd.DataFrame({"PPV": ppv, "SEN": sen, "SPC": spc, "F1": f1}, index=CLASS_NAMES) * 100
    overall = {"PPV": 100 * ppv.mean(), "SEN": 100 * sen.mean(), "SPC": 100 * spc.mean(),
               "F1": 100 * f1.mean(), "ACC": 100 * acc, "Error": 100 * (1 - acc)}
    return overall, per_class


pred_df = pd.read_csv(PRED_CSV)
y_true, y_pred = pred_df["y_true"].to_numpy(), pred_df["y_pred"].to_numpy()
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
overall, per_class = metrics_from_cm(cm)
per_class.round(2).to_csv(os.path.join(RESULTS_DIR, "per_class_metrics.csv"))

## 95% Confidence Intervals

Non-parametric bootstrap over the test images.

In [ ]:
# ==========================================================
# BOOTSTRAP 95% CONFIDENCE INTERVALS
# ==========================================================

rng = np.random.default_rng(RANDOM_SEED)
boot = {k: [] for k in ["PPV", "SEN", "SPC", "F1", "ACC"]}
for _ in range(BOOTSTRAP_ITERS):
    idx = rng.integers(0, len(y_true), len(y_true))
    m, _ = metrics_from_cm(confusion_matrix(y_true[idx], y_pred[idx], labels=range(len(CLASS_NAMES))))
    for k in boot:
        boot[k].append(m[k])

table = pd.DataFrame({
    "Value (%)": [overall[k] for k in ["PPV", "SEN", "SPC", "F1", "ACC", "Error"]],
    "95% CI": [f"{np.percentile(boot[k], 2.5):.2f}-{np.percentile(boot[k], 97.5):.2f}"
               for k in ["PPV", "SEN", "SPC", "F1", "ACC"]] + ["-"],
}, index=["PPV", "Recall/SEN", "SPC", "F1", "ACC", "Error"]).round(2)
table.to_csv(os.path.join(RESULTS_DIR, "classification_metrics.csv"))
print(table)

## Confusion Matrix

Rows represent actual classes and columns represent predicted classes.

In [ ]:
# ==========================================================
# CONFUSION MATRIX PLOT
# ==========================================================

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set_xlabel("Predicted class"); ax.set_ylabel("Actual class")
ax.set_title("Confusion Matrix - VGG16-DCNN")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=300)
plt.show()